In [1]:
import sys
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

print("Python executable:", sys.executable)
print("Pandas version:", pd.__version__)
print("NetworkX version:", nx.__version__)
print("NumPy version:", np.__version__)

Python executable: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\.venv\Scripts\python.exe
Pandas version: 3.0.6
NetworkX version: 3.6.1
NumPy version: 2.4.6


In [3]:
from pathlib import Path

# Find the project root, even if the notebook runs from src/
current_dir = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [current_dir, *current_dir.parents]
     if p.name == "adaptive-exam-timetabling"),
    current_dir
)

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "Nott"

print("Current directory:", current_dir)
print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATA_DIR)
print("Dataset directory exists:", DATA_DIR.exists())

if DATA_DIR.exists():
    print("\nFiles found:")
    for file_path in sorted(DATA_DIR.iterdir()):
        print(file_path.name)
else:
    print("\nDataset folder not found. Let's check the data/raw folders:")
    for folder in [PROJECT_ROOT / "data", PROJECT_ROOT / "data" / "raw"]:
        print(f"{folder} -> exists: {folder.exists()}")
        if folder.exists():
            print([p.name for p in folder.iterdir()])

Current directory: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\src
Project root: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling
Dataset directory: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\raw\Nott
Dataset directory exists: True

Files found:
data
enrolements
exams
README
students


In [4]:
from pathlib import Path

readme_path = DATA_DIR / "README"

print("README exists:", readme_path.exists())

if readme_path.exists():
    print(readme_path.read_text(encoding="utf-8", errors="replace"))
else:
    print("README not found. Checking filenames:")
    for p in DATA_DIR.rglob("*"):
        if p.is_file() and "readme" in p.name.lower():
            print(p)

README exists: True
This directory contains the main exam timetabling data for
the University of Nottingham, semester 1, 1994-95.
There are 800 exams provided by 46 departments, and 7896 students on 223
courses.  There are 33997 enrolements, so each student is sitting an average
of 4.3 exams, and each exam is sat by an average of 42.5 students.

The data is in the following files:

students      the list of students, with their courses;

                 student code      10 characters
                 course code        4     "

exams         the list of exams, containing the following fields;

                 exam code          8 characters
                 description       40     "
                 duration           4     "
                 department code    2     "

enrolements   a list of exams for each student;

                 student code      10 characters
                 exam code          8     "

data          other data, and constraints



In [2]:
from pathlib import Path

# Set the dataset path directly
PROJECT_ROOT = Path.cwd()

while (
    PROJECT_ROOT.name != "adaptive-exam-timetabling"
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "Nott"

print("Dataset directory:", DATA_DIR)
print("Dataset directory exists:", DATA_DIR.exists())

files_to_inspect = [
    "students",
    "exams",
    "enrolements",
    "data"
]

for filename in files_to_inspect:
    file_path = DATA_DIR / filename

    print("\n" + "=" * 60)
    print(f"FILE: {filename}")
    print("=" * 60)

    if not file_path.exists():
        print("File not found.")
        continue

    if file_path.is_file():
        print(f"Size: {file_path.stat().st_size:,} bytes")

        with file_path.open(
            "r", encoding="utf-8", errors="replace"
        ) as f:
            for line_number, line in enumerate(f):
                if line_number >= 5:
                    break
                print(f"{line_number + 1}: {line.rstrip()!r}")

    else:
        print("Directory contents:")
        for item in sorted(file_path.iterdir()):
            print(item.name)

Dataset directory: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\raw\Nott
Dataset directory exists: True

FILE: students
Size: 126,336 bytes
1: 'A890186790 R100'
2: 'A891097581 R200'
3: 'A892884191 Q400'
4: 'A892884227 Q400'
5: 'A892884294 V700'

FILE: exams
Size: 46,400 bytes
1: 'AA2016E1 OPERA STUDIES, I                         1:30 GM'
2: 'AA3008E1 HOLLYWOOD & THE EUROPEAN CINEMA          3:00 AI'
3: 'AAA011E1 INTRODUCTION TO ISLAM                    1:30 TH'
4: 'AAA013E1 INTRODUCTION TO SOCIAL ANTHROPOLOGY      2:00 TH'
5: 'AAA022E1 THE SOVIET POLITICAL SYSTEM              1:30 HI'

FILE: enrolements
Size: 679,940 bytes
1: 'A890186790 R13001E1'
2: 'A890186790 R13006E1'
3: 'A890186790 R13016E1'
4: 'A890186790 R13021E1'
5: 'A890186790 R13022E1'

FILE: data
Size: 3,770 bytes
1: 'DATES'
2: '-----'
3: 'Mon 23rd Jan - Sat 4th Feb 1995'
4: ''
5: 'TIMES'


In [3]:

from pathlib import Path

PROJECT_ROOT = Path.cwd()
while (
    PROJECT_ROOT.name != "adaptive-exam-timetabling"
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "Nott"

# 1. Read all scheduling constraints and information
print("=" * 65)
print("DATA FILE: SCHEDULING INFORMATION AND CONSTRAINTS")
print("=" * 65)

data_text = (DATA_DIR / "data").read_text(
    encoding="utf-8", errors="replace"
)
print(data_text)

# 2. Count non-empty records in each main file
print("\n" + "=" * 65)
print("RECORD COUNT VALIDATION")
print("=" * 65)

for filename in ["students", "exams", "enrolements"]:
    file_path = DATA_DIR / filename
    lines = [
        line for line in file_path.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()
        if line.strip()
    ]
    print(f"{filename:15} : {len(lines):,} records")

# 3. Check for duplicate raw records
print("\n" + "=" * 65)
print("DUPLICATE RECORD CHECK")
print("=" * 65)

for filename in ["students", "exams", "enrolements"]:
    lines = [
        line for line in (DATA_DIR / filename).read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()
        if line.strip()
    ]
    duplicates = len(lines) - len(set(lines))
    print(f"{filename:15} : {duplicates:,} duplicate lines")


DATA FILE: SCHEDULING INFORMATION AND CONSTRAINTS
DATES
-----
Mon 23rd Jan - Sat 4th Feb 1995

TIMES
-----
Mon - Fri  9:00 (3hrs), 13:30 (2hrs), 16:30 (2hrs)
Sat        9:00 (3hrs)

ROOMS
-----
TRENT-HALL   125
TRENT-L19     80
TRENT-B46     40
COPSE-2       40
COPSE-3       50
COPSE-4       35
COPSE-5       30
COPSE-6       25
PRTLND-GRND  200
POPE-A13      80 \ together  \
POPE-A14      80 /           |  Rooms that are together can be combined,
ART-LECTURE   80 \ together  -  ie. used as one room.  This is not as good as
ART-SEMINAR   15 /           |  using one room, but is better than splitting
SPORT-LGE1   250 \ together  |  an exam over two distant rooms.
SPORT-LGE2   230 /           /
SPORT-SMALL  270

ROOM ASSIGNMENTS
----------------
V41103E1   ART-SEMINAR
V41112E1   ART-LECTURE
V41124E1   ART-SEMINAR
V41133E1   ART-LECTURE
V41142E1   ART-LECTURE
V41143E1   ART-SEMINAR
V42103E1   ART-LECTURE
V42112E1   ART-LECTURE
V42117E1   ART-LECTURE
V42124E1   ART-LECTURE
V42133E1   ART-LE

In [4]:

from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path.cwd()
while (
    PROJECT_ROOT.name != "adaptive-exam-timetabling"
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "Nott"

# Read the original records without changing the files
student_lines = [
    line.strip()
    for line in (DATA_DIR / "students").read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()
    if line.strip()
]

exam_lines = [
    line.strip()
    for line in (DATA_DIR / "exams").read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()
    if line.strip()
]

enrolment_lines = [
    line.strip()
    for line in (DATA_DIR / "enrolements").read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()
    if line.strip()
]

# Parse student and enrolment records
students = {}
for line in student_lines:
    parts = line.split()
    if len(parts) == 2:
        students[parts[0]] = parts[1]

exam_ids = set()
for line in exam_lines:
    exam_id = line[:8].strip()
    if exam_id:
        exam_ids.add(exam_id)

student_exam_pairs = []
for line in enrolment_lines:
    parts = line.split()
    if len(parts) == 2:
        student_exam_pairs.append((parts[0], parts[1]))

enrolled_student_ids = {s for s, e in student_exam_pairs}
enrolled_exam_ids = {e for s, e in student_exam_pairs}

# Relationship validation
unknown_students = enrolled_student_ids - set(students)
unknown_exams = enrolled_exam_ids - exam_ids

pair_counts = Counter(student_exam_pairs)
duplicate_pairs = sum(count - 1 for count in pair_counts.values())

student_exam_counts = Counter(s for s, e in student_exam_pairs)

print("STUDENT–EXAM RELATIONSHIP VALIDATION")
print("-" * 45)
print("Unique student IDs parsed:", len(students))
print("Unique exam IDs parsed:", len(exam_ids))
print("Valid enrolment pairs parsed:", len(student_exam_pairs))
print("Enrolments referencing unknown students:", len(unknown_students))
print("Enrolments referencing unknown exams:", len(unknown_exams))
print("Repeated student–exam pairs:", duplicate_pairs)

print("\nEXAMS PER STUDENT")
print("-" * 45)
print("Students with at least one enrolment:", len(student_exam_counts))

if student_exam_counts:
    print(
        "Average exams per enrolled student:",
        round(sum(student_exam_counts.values()) / len(student_exam_counts), 3)
    )
    print("Minimum exams:", min(student_exam_counts.values()))
    print("Maximum exams:", max(student_exam_counts.values()))

print("\nSample student–exam pairs:")
for pair in student_exam_pairs[:10]:
    print(pair)


STUDENT–EXAM RELATIONSHIP VALIDATION
---------------------------------------------
Unique student IDs parsed: 7896
Unique exam IDs parsed: 800
Valid enrolment pairs parsed: 33997
Enrolments referencing unknown students: 0
Enrolments referencing unknown exams: 0
Repeated student–exam pairs: 0

EXAMS PER STUDENT
---------------------------------------------
Students with at least one enrolment: 7896
Average exams per enrolled student: 4.306
Minimum exams: 1
Maximum exams: 9

Sample student–exam pairs:
('A890186790', 'R13001E1')
('A890186790', 'R13006E1')
('A890186790', 'R13016E1')
('A890186790', 'R13021E1')
('A890186790', 'R13022E1')
('A891097581', 'R23102E1')
('A891097581', 'R23107E1')
('A891097581', 'R23111E1')
('A891097581', 'R23117E1')
('A891097581', 'R23209E1')


In [5]:

import networkx as nx
from itertools import combinations
from collections import defaultdict

# Reuse the validated student_exam_pairs and exam_ids
# Create a graph containing every examination
G = nx.Graph()
G.add_nodes_from(sorted(exam_ids))

# Group examinations by student
student_to_exams = defaultdict(list)

for student_id, exam_id in student_exam_pairs:
    student_to_exams[student_id].append(exam_id)

# Connect every pair of exams taken by the same student
for exams_for_student in student_to_exams.values():
    for exam_a, exam_b in combinations(exams_for_student, 2):
        G.add_edge(exam_a, exam_b)

# Graph statistics
print("CONFLICT GRAPH SUMMARY")
print("-" * 45)
print("Number of exam vertices:", G.number_of_nodes())
print("Number of conflict edges:", G.number_of_edges())
print("Number of connected components:", nx.number_connected_components(G))
print("Is the graph connected:", nx.is_connected(G))
print("Maximum degree:", max(dict(G.degree()).values()))
print("Average degree:", round(sum(dict(G.degree()).values()) / G.number_of_nodes(), 3))
print("Self-loops:", nx.number_of_selfloops(G))

# Inspect the exams with the most conflicts
print("\nTOP 10 EXAMS BY CONFLICT DEGREE")
print("-" * 45)

for exam_id, degree in sorted(
    G.degree(), key=lambda item: item[1], reverse=True
)[:10]:
    print(f"{exam_id}: {degree} conflicts")

# Save the graph for reuse in later notebooks
import pickle

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

graph_path = output_dir / "conflict_graph.gpickle"

# Use pickle to save the NetworkX graph
with graph_path.open("wb") as f:
    pickle.dump(G, f)

print("\nGraph saved to:", graph_path)


CONFLICT GRAPH SUMMARY
---------------------------------------------
Number of exam vertices: 800
Number of conflict edges: 10113
Number of connected components: 5
Is the graph connected: False
Maximum degree: 203
Average degree: 25.282
Self-loops: 0

TOP 10 EXAMS BY CONFLICT DEGREE
---------------------------------------------
LKA1FAE1: 203 conflicts
LK10SAE1: 144 conflicts
LK10IAE1: 115 conflicts
LK10GAE1: 113 conflicts
LKB2FAE1: 112 conflicts
V41142E1: 112 conflicts
LKC3FAE1: 110 conflicts
LKA1GAE1: 108 conflicts
V7A107E1: 105 conflicts
L81122E1: 101 conflicts

Graph saved to: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\data\processed\conflict_graph.gpickle
